In [21]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu

np.random.seed(42)
import random
random.seed(42)


In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import os
os.environ["SCIPY_ARRAY_API"] = "1"
from imblearn.over_sampling import RandomOverSampler

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.cuda.is_available()


True

In [23]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# Import VAE models from functions module
import sys
sys.path.append('functions')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [24]:
import importlib
import functions.vae_models
# Now python knows what 'functions' is
importlib.reload(functions.vae_models)
from functions.vae_models import TransformerMultiModalConditionalVAE, SingleModalConditionalVAE


In [25]:
data_folder = './data/EAE/attn/raw_cdr3_5-30/'

# Create results folder at the same level as data_folder
results_folder = 'results_transformer/'
os.makedirs(results_folder, exist_ok=True)
print(f"Results will be saved to: {results_folder}")


Results will be saved to: results_transformer/


In [26]:
# Load CSV files
tcr_embs = pd.read_csv(data_folder + "tcr_embs.csv", index_col=0)
gex_df = pd.read_csv(data_folder + "gex_df.csv", index_col=0)
labels = pd.read_csv(data_folder + "labels.csv", index_col=0)

print(f"TCR embeddings shape: {tcr_embs.shape}")
print(f"Gene expression shape: {gex_df.shape}")
print(f"Labels shape: {labels.shape}")
print(f"\nTissue classes: {labels['label_tissue'].unique()}")
print(f"Sample IDs: {labels['label_sample_id'].nunique()} unique samples")


TCR embeddings shape: (106241, 486)
Gene expression shape: (106241, 2995)
Labels shape: (106241, 7)

Tissue classes: ['CNS' 'Spleen' 'iLN' nan 'mLN' 'MLN' 'COL' 'SI' 'PP' 'DLN']
Sample IDs: 19 unique samples


## Label process


In [27]:
# Prepare data
# Filter out rows with NaN tissue labels
valid_mask = labels['label_tissue'].notna()
print(f"Rows with valid tissue labels: {valid_mask.sum()} / {len(labels)}")

# Filter data
tcr_embs_filtered = tcr_embs.loc[valid_mask]
gex_df_filtered = gex_df.loc[valid_mask]
labels_filtered = labels.loc[valid_mask]
common_idx_filtered = labels_filtered.index

# Convert tissue labels to 3 classes: 'CNS', 'Spleen', and 'else'
def convert_tissue_label(tissue):
    if pd.isna(tissue):
        return 'else'
    tissue_str = str(tissue).strip()
    if tissue_str == 'CNS':
        return 'CNS'
    elif tissue_str == 'Spleen':
        return 'Spleen'
    else:
        return 'else'

labels_filtered = labels_filtered.copy()
labels_filtered['label_tissue_converted'] = labels_filtered['label_tissue'].apply(convert_tissue_label)

print(f"\nTissue label distribution after conversion:")
print(labels_filtered['label_tissue_converted'].value_counts())


Rows with valid tissue labels: 100480 / 106241

Tissue label distribution after conversion:
label_tissue_converted
Spleen    45310
CNS       41410
else      13760
Name: count, dtype: int64


In [ ]:
labels_filtered.head(5)


,label_tissue,label_cell_type,label_state,label_GSE,label_sample_id,label_set,label_clone_id_size,label_tissue_converted
AAGTAGCAGATAGGCG-1_0516_CNS,CNS,CD4,Activation,LEE,5_7,train,1.0,CNS
AAGTATACACCCAGTA-1_0516_CNS,CNS,NaN,Activation,LEE,5_7,train,1.0,CNS
AAGTTTGGTGGAACCC-1_0516_CNS,CNS,NaN,Exhaust,LEE,5_3,train,1.0,CNS
AATGCGACACTAAAGG-1_0516_CNS,CNS,CD8,NaN,LEE,5_7,train,1.0,CNS
AATGCGACACTATGAC-1_0516_CNS,CNS,NaN,Activation,LEE,5_3,train,3.0,CNS


## Select Features and Target Label


In [ ]:
target_label = 'label_tissue_converted'
sample_label = 'label_GSE'


In [ ]:
# Encode sample id labels as one-hot
sample_id_encoder = LabelEncoder()
std_scaler = StandardScaler()
sample_id_encoded = sample_id_encoder.fit_transform(labels_filtered[sample_label])
n_targer_class = len(sample_id_encoder.classes_)
sample_id_onehot = np.eye(n_targer_class)[sample_id_encoded]

# Add numeric values in labels_filtered as features
numeric_features = labels_filtered[['label_clone_id_size']].values.astype(float)
# 'label_CV_score_0', 'label_CV_score_1','label_CV_score_2'
numeric_features = std_scaler.fit_transform(numeric_features)

# Concatenate sample_id_onehot with clone_id_size to create condition features
sample_id_onehot = np.hstack([numeric_features, sample_id_onehot])


# Encode target labels
label_target = labels_filtered[target_label]
tissue_encoder = LabelEncoder()
tissue_labels = tissue_encoder.fit_transform(label_target)
n_tissues = len(tissue_encoder.classes_)

# Normalize features

tcr_data = std_scaler.fit_transform(tcr_embs_filtered.values)
gex_data = std_scaler.fit_transform(gex_df_filtered.values)

print(f"\nTCR data shape: {tcr_data.shape}")
print(f"GEX data shape: {gex_data.shape}")
print(f"Sample ID one-hot shape: {sample_id_onehot.shape}")
print(f"Number of tissue classes: {n_tissues}")
print(f"Tissue classes: {tissue_encoder.classes_}")
print(f"\nLabel set distribution:")
print(labels_filtered['label_set'].value_counts())



TCR data shape: (100480, 486)
GEX data shape: (100480, 2995)
Sample ID one-hot shape: (100480, 7)
Number of tissue classes: 3
Tissue classes: ['CNS' 'Spleen' 'else']

Label set distribution:
label_set
train    90440
test     10040
Name: count, dtype: int64


## Train-Test Split


In [ ]:
# Split data: use label_set to separate train and test
# Exclude label_set=='test' from training, use only for final evaluation
# CRITICAL: Ensure both train and test sets have all classes

train_mask = labels_filtered['label_set'] != 'test'
test_mask = labels_filtered['label_set'] == 'test'

train_idx = np.where(train_mask)[0]
test_idx = np.where(test_mask)[0]

print(f"Initial split - Train size (label_set != 'test'): {len(train_idx)}")
print(f"Initial split - Test size (label_set == 'test'): {len(test_idx)}")

# Check which classes are present in test set
test_classes = np.unique(tissue_labels[test_idx]) if len(test_idx) > 0 else np.array([])
all_classes = np.arange(n_tissues)
missing_classes = np.setdiff1d(all_classes, test_classes)

print(f"\nClasses in test set: {[tissue_encoder.inverse_transform([c])[0] for c in test_classes]}")
print(f"Missing classes in test set: {[tissue_encoder.inverse_transform([c])[0] for c in missing_classes]}")

# If test set is missing some classes, move samples from train to test
if len(missing_classes) > 0 and len(test_idx) > 0:
    print(f"\nMoving samples from train to test to ensure all classes are present...")
    samples_to_move = []
    
    for missing_class in missing_classes:
        # Find samples of this class in train set
        train_class_mask = tissue_labels[train_idx] == missing_class
        train_class_indices = train_idx[train_class_mask]
        
        if len(train_class_indices) > 0:
            # Move a small percentage (e.g., 5-10%) of this class to test
            n_to_move = max(1, min(10, len(train_class_indices) // 10))
            np.random.seed(42)
            selected = np.random.choice(train_class_indices, size=n_to_move, replace=False)
            samples_to_move.extend(selected)
            print(f"  Moving {n_to_move} samples of class '{tissue_encoder.inverse_transform([missing_class])[0]}' to test")
    
    if len(samples_to_move) > 0:
        samples_to_move = np.array(samples_to_move)
        # Update indices
        train_idx = np.setdiff1d(train_idx, samples_to_move)
        test_idx = np.concatenate([test_idx, samples_to_move])
        print(f"  Moved {len(samples_to_move)} samples total")

elif len(test_idx) == 0:
    # If no test set, split from train data with stratification
    print("Warning: No rows with label_set=='test' found. Splitting train data instead.")
    train_idx, test_idx = train_test_split(
        train_idx,
        test_size=0.2,
        random_state=42,
        stratify=tissue_labels[train_idx]
    )

# Final check: ensure both sets have all classes
train_classes = np.unique(tissue_labels[train_idx])
test_classes = np.unique(tissue_labels[test_idx])

print(f"\nFinal class check:")
print(f"  Train classes: {[tissue_encoder.inverse_transform([c])[0] for c in train_classes]}")
print(f"  Test classes: {[tissue_encoder.inverse_transform([c])[0] for c in test_classes]}")

if len(train_classes) < n_tissues or len(test_classes) < n_tissues:
    print("WARNING: Not all classes are present in both sets!")
    print("This may cause issues with classification_report. Consider adjusting the split.")

# Create final splits
tcr_train, tcr_test = tcr_data[train_idx], tcr_data[test_idx]
gex_train, gex_test = gex_data[train_idx], gex_data[test_idx]
sample_id_train, sample_id_test = sample_id_onehot[train_idx], sample_id_onehot[test_idx]
tissue_train, tissue_test = tissue_labels[train_idx], tissue_labels[test_idx]

print(f"\nFinal train size: {len(train_idx)}, Test size: {len(test_idx)}")
print(f"Train tissue distribution:")
unique_train, counts_train = np.unique(tissue_train, return_counts=True)
for label_idx, count in zip(unique_train, counts_train):
    print(f"  {tissue_encoder.inverse_transform([label_idx])[0]}: {count}")
print(f"Test tissue distribution:")
unique_test, counts_test = np.unique(tissue_test, return_counts=True)
for label_idx, count in zip(unique_test, counts_test):
    print(f"  {tissue_encoder.inverse_transform([label_idx])[0]}: {count}")


Initial split - Train size (label_set != 'test'): 90440
Initial split - Test size (label_set == 'test'): 10040

Classes in test set: ['CNS', 'Spleen']
Missing classes in test set: ['else']

Moving samples from train to test to ensure all classes are present...
  Moving 10 samples of class 'else' to test
  Moved 10 samples total

Final class check:
  Train classes: ['CNS', 'Spleen', 'else']
  Test classes: ['CNS', 'Spleen', 'else']

Final train size: 90430, Test size: 10050
Train tissue distribution:
  CNS: 38220
  Spleen: 38460
  else: 13750
Test tissue distribution:
  CNS: 3190
  Spleen: 6850
  else: 10


In [ ]:
# Condition encoder: TransformerMultiModalConditionalVAE expects condition in condition_emb_dim
# We need to encode the condition from cond_in_dim to condition_emb_dim
cond_in_dim = sample_id_onehot.shape[1]
condition_emb_dim = 10

# Create condition encoder
condition_encoder = nn.Sequential(
    nn.Linear(cond_in_dim, condition_emb_dim),
    nn.ReLU(),
    nn.BatchNorm1d(condition_emb_dim),
).to(device)

# Initialize model
model = TransformerMultiModalConditionalVAE(
    tcr_dim=tcr_data.shape[1],
    gex_dim=gex_data.shape[1],
    condition_emb_dim=condition_emb_dim,
    latent_dim=128,
    hidden_dim=512,
    n_classes=n_tissues,
    transformer_d_model=512,
    transformer_nhead=8,
    transformer_num_layers=2,
    transformer_dim_feedforward=1024,
    transformer_dropout=0.1
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Condition encoder parameters: {sum(p.numel() for p in condition_encoder.parameters()):,}")
print(model)


Model parameters: 25,503,132
Condition encoder parameters: 100
TransformerMultiModalConditionalVAE(
  (tcr_proj): Linear(in_features=486, out_features=512, bias=True)
  (tcr_pos_encoding): PositionalEncoding()
  (tcr_transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (gex_proj): Linear(in_features=2995, out_features=512, bias=True)
  (gex_

In [ ]:
model.tcr_transformer.layers[0].linear1

Linear(in_features=512, out_features=1024, bias=True)

In [ ]:
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"{name:40s} {tuple(p.shape)}")


tcr_proj.weight                          (512, 486)
tcr_proj.bias                            (512,)
tcr_transformer.layers.0.self_attn.in_proj_weight (1536, 512)
tcr_transformer.layers.0.self_attn.in_proj_bias (1536,)
tcr_transformer.layers.0.self_attn.out_proj.weight (512, 512)
tcr_transformer.layers.0.self_attn.out_proj.bias (512,)
tcr_transformer.layers.0.linear1.weight  (1024, 512)
tcr_transformer.layers.0.linear1.bias    (1024,)
tcr_transformer.layers.0.linear2.weight  (512, 1024)
tcr_transformer.layers.0.linear2.bias    (512,)
tcr_transformer.layers.0.norm1.weight    (512,)
tcr_transformer.layers.0.norm1.bias      (512,)
tcr_transformer.layers.0.norm2.weight    (512,)
tcr_transformer.layers.0.norm2.bias      (512,)
tcr_transformer.layers.1.self_attn.in_proj_weight (1536, 512)
tcr_transformer.layers.1.self_attn.in_proj_bias (1536,)
tcr_transformer.layers.1.self_attn.out_proj.weight (512, 512)
tcr_transformer.layers.1.self_attn.out_proj.bias (512,)
tcr_transformer.layers.1.linear1.

In [ ]:
## freeze
for param in model.parameters(): 
      # param.requires_grad = False

Parameter containing:
tensor([[-4.9885e-40, -4.9278e-40, -2.5404e-02,  ...,  1.2225e-02,
         -5.2313e-03, -3.5539e-02],
        [ 4.9606e-40,  4.9350e-40, -1.5556e-02,  ..., -3.1460e-02,
          7.9815e-03,  2.4267e-03],
        [ 4.9439e-40, -4.9065e-40, -7.5139e-03,  ...,  1.0570e-03,
          1.8915e-02, -3.4730e-02],
        ...,
        [ 4.9426e-40, -4.9291e-40, -2.5300e-02,  ..., -2.7222e-02,
         -1.0891e-02, -2.5075e-02],
        [-4.9401e-40,  4.9078e-40, -2.7795e-03,  ...,  8.5377e-02,
          2.8294e-03,  5.5778e-03],
        [-4.9380e-40, -4.9238e-40,  2.6520e-02,  ...,  3.0869e-02,
         -3.0552e-02,  7.4156e-02]], device='cuda:0', requires_grad=True)
Parameter containing:
tensor([ 1.5149e-02, -3.7142e-02,  4.1877e-02, -1.2396e-02,  3.1808e-02,
        -6.2153e-02,  2.6480e-02,  1.9121e-02,  4.3254e-02, -3.6017e-02,
         5.3704e-02,  7.7119e-03,  2.3127e-02, -3.0114e-02,  4.7591e-02,
        -3.2028e-02,  7.9944e-03, -4.5859e-02, -8.4284e-03, -1.4152e

In [ ]:
# Loss functions
def vae_loss(tcr_recon, tcr_true, gex_recon, gex_true, mu, logvar, 
             tcr_weight=1.0, gex_weight=1.0, kl_weight=0.001):
    """VAE loss: reconstruction + KL divergence"""
    # Reconstruction losses (MSE)
    tcr_recon_loss = F.mse_loss(tcr_recon, tcr_true, reduction='mean')
    gex_recon_loss = F.mse_loss(gex_recon, gex_true, reduction='mean')
    
    # KL divergence
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    
    total_loss = (tcr_weight * tcr_recon_loss + 
                  gex_weight * gex_recon_loss + 
                  kl_weight * kl_loss)
    
    return total_loss, tcr_recon_loss, gex_recon_loss, kl_loss

def classification_loss(pred, target):
    """Cross-entropy loss for classification"""
    return F.cross_entropy(pred, target)


In [34]:
# Create data loaders
batch_size = 64

train_dataset = TensorDataset(
    torch.FloatTensor(tcr_train),
    torch.FloatTensor(gex_train),
    torch.FloatTensor(sample_id_train),
    torch.LongTensor(tissue_train)
)

test_dataset = TensorDataset(
    torch.FloatTensor(tcr_test),
    torch.FloatTensor(gex_test),
    torch.FloatTensor(sample_id_test),
    torch.LongTensor(tissue_test)
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")


Train batches: 1413, Test batches: 158


In [35]:
# Create data loaders
batch_size = 64

train_dataset = TensorDataset(
    torch.FloatTensor(tcr_train),
    torch.FloatTensor(gex_train),
    torch.FloatTensor(sample_id_train),
    torch.LongTensor(tissue_train)
)

test_dataset = TensorDataset(
    torch.FloatTensor(tcr_test),
    torch.FloatTensor(gex_test),
    torch.FloatTensor(sample_id_test),
    torch.LongTensor(tissue_test)
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")


Train batches: 1413, Test batches: 158


In [ ]:
# Training setup - Grid search parameters
n_epochs = 20

# Define lists for grid search
vae_weight_list = [0.3, 1.0]
class_weight_list = [1.0, 0.3]

print(f"Grid search will test {len(vae_weight_list) * len(class_weight_list)} combinations:")
print(f"  vae_weight values: {vae_weight_list}")
print(f"  class_weight values: {class_weight_list}")


Grid search will test 4 combinations:
  vae_weight values: [0.3, 1.0]
  class_weight values: [1.0, 0.3]


In [ ]:
# Grid search over vae_weight and class_weight combinations (paired by index)
import copy

# Ensure lists have the same length
min_len = min(len(vae_weight_list), len(class_weight_list))
vae_weight_list = vae_weight_list[:min_len]
class_weight_list = class_weight_list[:min_len]

grid_search_results = []
best_combination = None
best_test_acc_grid = 0.0

total_combinations = len(vae_weight_list)
print(f"Starting grid search over {total_combinations} combinations (paired by index)...\n")

for combo_idx, (vae_weight, class_weight) in enumerate(zip(vae_weight_list, class_weight_list), 1):
    print(f"\n{'='*60}")
    print(f"Combination {combo_idx}/{total_combinations}: vae_weight={vae_weight}, class_weight={class_weight}")
    print(f"{'='*60}")
    
    # Check if model already exists
    model_filename = f'vae_transformer_vae{vae_weight}_class{class_weight}.pth'
    model_filepath = os.path.join(results_folder, model_filename)
    
    # Reinitialize model and condition encoder for each combination
    model = TransformerMultiModalConditionalVAE(
        tcr_dim=tcr_data.shape[1],
        gex_dim=gex_data.shape[1],
        condition_emb_dim=condition_emb_dim,
        latent_dim=128,
        hidden_dim=512,
        n_classes=n_tissues,
        transformer_d_model=512,
        transformer_nhead=8,
        transformer_num_layers=2,
        transformer_dim_feedforward=1024,
        transformer_dropout=0.1
    ).to(device)
    
    condition_encoder = nn.Sequential(
        nn.Linear(cond_in_dim, condition_emb_dim),
        nn.ReLU(),
        nn.BatchNorm1d(condition_emb_dim),
    ).to(device)
    
    # Check if model exists and load it
    if os.path.exists(model_filepath):
        print(f"  -> Loading pre-trained model from: {model_filepath}")
        checkpoint = torch.load(model_filepath, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        condition_encoder.load_state_dict(checkpoint['condition_encoder_state_dict'])
        best_test_acc_combo = checkpoint.get('best_test_acc', 0.0)
        print(f"  -> Loaded model with best test accuracy: {best_test_acc_combo:.2f}%")
        
        # Re-evaluate to get current test accuracy
        model.eval()
        condition_encoder.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for tcr_batch, gex_batch, condition_batch, tissue_batch in test_loader:
                tcr_batch = tcr_batch.to(device)
                gex_batch = gex_batch.to(device)
                condition_batch = condition_batch.to(device)
                tissue_batch = tissue_batch.to(device)
                
                # Encode condition
                condition_encoded = condition_encoder(condition_batch)
                
                tcr_recon, gex_recon, mu, logvar, z, tissue_pred = model(
                    tcr_batch, gex_batch, condition_encoded
                )
                
                _, predicted = torch.max(tissue_pred.data, 1)
                test_total += tissue_batch.size(0)
                test_correct += (predicted == tissue_batch).sum().item()
        
        test_acc = 100 * test_correct / test_total
        best_test_acc_combo = test_acc  # Update with current evaluation
        print(f"  -> Current test accuracy: {test_acc:.2f}%")
    else:
        print(f"  -> Model not found. Training new model...")
        # Training setup for this combination
        optimizer = optim.Adam(
            list(model.parameters()) + list(condition_encoder.parameters()), 
            lr=0.0001, 
            weight_decay=1e-5
        )
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=False)
        
        best_test_acc_combo = 0.0
        
        # Training loop for this combination
        for epoch in range(n_epochs):
            # Training
            model.train()
            condition_encoder.train()
            train_loss = 0.0
            train_vae_loss = 0.0
            train_class_loss = 0.0
            train_correct = 0
            train_total = 0
            
            for tcr_batch, gex_batch, condition_batch, tissue_batch in train_loader:
                tcr_batch = tcr_batch.to(device)
                gex_batch = gex_batch.to(device)
                condition_batch = condition_batch.to(device)
                tissue_batch = tissue_batch.to(device)
                
                optimizer.zero_grad()
                
                # Encode condition
                condition_encoded = condition_encoder(condition_batch)
                
                # Forward pass
                tcr_recon, gex_recon, mu, logvar, z, tissue_pred = model(
                    tcr_batch, gex_batch, condition_encoded
                )
                
                # Losses
                vae_loss_val, tcr_recon_loss, gex_recon_loss, kl_loss = vae_loss(
                    tcr_recon, tcr_batch, gex_recon, gex_batch, mu, logvar
                )
                class_loss_val = classification_loss(tissue_pred, tissue_batch)
                
                total_loss = vae_weight * vae_loss_val + class_weight * class_loss_val
                
                # Backward pass
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                torch.nn.utils.clip_grad_norm_(condition_encoder.parameters(), max_norm=1.0)
                optimizer.step()
                
                # Metrics
                train_loss += total_loss.item()
                train_vae_loss += vae_loss_val.item()
                train_class_loss += class_loss_val.item()
                
                _, predicted = torch.max(tissue_pred.data, 1)
                train_total += tissue_batch.size(0)
                train_correct += (predicted == tissue_batch).sum().item()
            
            # Validation
            model.eval()
            condition_encoder.eval()
            test_loss = 0.0
            test_vae_loss = 0.0
            test_class_loss = 0.0
            test_correct = 0
            test_total = 0
            
            with torch.no_grad():
                for tcr_batch, gex_batch, condition_batch, tissue_batch in test_loader:
                    tcr_batch = tcr_batch.to(device)
                    gex_batch = gex_batch.to(device)
                    condition_batch = condition_batch.to(device)
                    tissue_batch = tissue_batch.to(device)
                    
                    # Encode condition
                    condition_encoded = condition_encoder(condition_batch)
                    
                    tcr_recon, gex_recon, mu, logvar, z, tissue_pred = model(
                        tcr_batch, gex_batch, condition_encoded
                    )
                    
                    vae_loss_val, _, _, _ = vae_loss(
                        tcr_recon, tcr_batch, gex_recon, gex_batch, mu, logvar
                    )
                    class_loss_val = classification_loss(tissue_pred, tissue_batch)
                    
                    total_loss = vae_weight * vae_loss_val + class_weight * class_loss_val
                    
                    test_loss += total_loss.item()
                    test_vae_loss += vae_loss_val.item()
                    test_class_loss += class_loss_val.item()
                    
                    _, predicted = torch.max(tissue_pred.data, 1)
                    test_total += tissue_batch.size(0)
                    test_correct += (predicted == tissue_batch).sum().item()
            
            # Average metrics
            train_loss /= len(train_loader)
            train_vae_loss /= len(train_loader)
            train_class_loss /= len(train_loader)
            train_acc = 100 * train_correct / train_total
            
            test_loss /= len(test_loader)
            test_vae_loss /= len(test_loader)
            test_class_loss /= len(test_loader)
            test_acc = 100 * test_correct / test_total
            
            scheduler.step(test_loss)
            
            if test_acc > best_test_acc_combo:
                best_test_acc_combo = test_acc
            
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Epoch [{epoch+1}/{n_epochs}] - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
        
        # Save model for this combination (only if it was trained)
        torch.save({
            'model_state_dict': model.state_dict(),
            'condition_encoder_state_dict': condition_encoder.state_dict(),
            'vae_weight': vae_weight,
            'class_weight': class_weight,
            'best_test_acc': best_test_acc_combo
        }, model_filepath)
        print(f"  -> Model saved to: {model_filepath}")
    
    # Store results for this combination
    grid_search_results.append({
        'vae_weight': vae_weight,
        'class_weight': class_weight,
        'best_test_acc': best_test_acc_combo,
        'model_filepath': model_filepath
    })
    
    print(f"  -> Best test accuracy: {best_test_acc_combo:.2f}%")
    
    # Track best combination overall
    if best_test_acc_combo > best_test_acc_grid:
        best_test_acc_grid = best_test_acc_combo
        best_combination = {
            'vae_weight': vae_weight,
            'class_weight': class_weight,
            'best_test_acc': best_test_acc_combo,
            'model_filepath': model_filepath
        }

print(f"\n{'='*60}")
print("Grid search completed!")
print(f"{'='*60}")
print("\nGrid search results:")
results_df = pd.DataFrame(grid_search_results)
results_df = results_df.sort_values('best_test_acc', ascending=False)
print(results_df.to_string(index=False))
print(f"\nBest combination: vae_weight={best_combination['vae_weight']}, class_weight={best_combination['class_weight']}")
print(f"Best test accuracy: {best_test_acc_grid:.2f}%")


Starting grid search over 2 combinations (paired by index)...


Combination 1/2: vae_weight=0.3, class_weight=1.0
  -> Model not found. Training new model...
  Epoch [1/20] - Test Acc: 94.77%
  Epoch [10/20] - Test Acc: 92.88%
  Epoch [20/20] - Test Acc: 95.43%
  -> Model saved to: results_transformer/vae_transformer_vae0.3_class1.0.pth
  -> Best test accuracy: 95.49%

Combination 2/2: vae_weight=1.0, class_weight=0.3
  -> Model not found. Training new model...
  Epoch [1/20] - Test Acc: 94.68%
  Epoch [10/20] - Test Acc: 95.82%
  Epoch [20/20] - Test Acc: 95.68%
  -> Model saved to: results_transformer/vae_transformer_vae1.0_class0.3.pth
  -> Best test accuracy: 96.49%

Grid search completed!

Grid search results:
 vae_weight  class_weight  best_test_acc                                          model_filepath
        1.0           0.3      96.487562 results_transformer/vae_transformer_vae1.0_class0.3.pth
        0.3           1.0      95.492537 results_transformer/vae_transformer_vae0

In [ ]:
## Load best model from grid search
# Load the best model from grid search
vae_weight = best_combination['vae_weight']
class_weight = best_combination['class_weight']
best_model_path = best_combination['model_filepath']

print(f"Loading best model from grid search:")
print(f"  vae_weight={vae_weight}, class_weight={class_weight}")
print(f"  Model path: {best_model_path}")
print(f"  Best test accuracy from grid search: {best_test_acc_grid:.2f}%")

# Reinitialize model and condition encoder
model = TransformerMultiModalConditionalVAE(
    tcr_dim=tcr_data.shape[1],
    gex_dim=gex_data.shape[1],
    condition_emb_dim=condition_emb_dim,
    latent_dim=128,
    hidden_dim=512,
    n_classes=n_tissues,
    transformer_d_model=512,
    transformer_nhead=8,
    transformer_num_layers=2,
    transformer_dim_feedforward=1024,
    transformer_dropout=0.1
).to(device)

condition_encoder = nn.Sequential(
    nn.Linear(cond_in_dim, condition_emb_dim),
    nn.ReLU(),
    nn.BatchNorm1d(condition_emb_dim),
).to(device)

# Load best model from grid search
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
condition_encoder.load_state_dict(checkpoint['condition_encoder_state_dict'])

# Also save as the main best model for compatibility with evaluation cells
torch.save({
    'model_state_dict': model.state_dict(),
    'condition_encoder_state_dict': condition_encoder.state_dict(),
    'vae_weight': vae_weight,
    'class_weight': class_weight,
}, 'best_vae_transformer_model.pth')

print(f"\nBest model loaded and saved to 'best_vae_transformer_model.pth'")
print(f"Best test accuracy: {best_test_acc_grid:.2f}%")


Loading best model from grid search:
  vae_weight=1.0, class_weight=0.3
  Model path: results_transformer/vae_transformer_vae1.0_class0.3.pth
  Best test accuracy from grid search: 96.49%

Best model loaded and saved to 'best_vae_transformer_model.pth'
Best test accuracy: 96.49%


In [41]:
# Save grid search results
grid_search_df = pd.DataFrame(grid_search_results)
grid_search_df = grid_search_df.sort_values('best_test_acc', ascending=False)

# Save to CSV
grid_search_filepath = os.path.join(results_folder, 'grid_search_results.csv')
grid_search_df.to_csv(grid_search_filepath, index=False)
print(f"Grid search results saved to '{grid_search_filepath}'")
print("\nTop 5 combinations:")
print(grid_search_df.head(5).to_string(index=False))


Grid search results saved to 'results_transformer/grid_search_results.csv'

Top 5 combinations:
 vae_weight  class_weight  best_test_acc                                          model_filepath
        1.0           0.3      96.487562 results_transformer/vae_transformer_vae1.0_class0.3.pth
        0.3           1.0      95.492537 results_transformer/vae_transformer_vae0.3_class1.0.pth


In [ ]:
# Save combined model results
combined_model_results = {
    'model_name': 'Transformer Multi-modal (GEX + TCR)',
    'best_test_acc': best_test_acc_grid,
    'vae_weight': vae_weight,
    'class_weight': class_weight
}
print(f"Transformer Multi-modal model best test accuracy: {best_test_acc_grid:.2f}%")
print(f"Best weights from grid search: vae_weight={vae_weight}, class_weight={class_weight}")


## Evaluation Function


In [44]:
## Evaluation Function
def evaluate_model(model, condition_encoder, test_loader, model_name):
    """Evaluate model and return predictions with proper label handling"""
    model.eval()
    condition_encoder.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        # Transformer Multi-modal model
        for tcr_batch, gex_batch, condition_batch, tissue_batch in test_loader:
            tcr_batch = tcr_batch.to(device)
            gex_batch = gex_batch.to(device)
            condition_batch = condition_batch.to(device)
            
            # Encode condition
            condition_encoded = condition_encoder(condition_batch)
            
            _, _, _, _, _, tissue_pred = model(tcr_batch, gex_batch, condition_encoded)
            _, predicted = torch.max(tissue_pred, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(tissue_batch.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Get all possible class labels (0 to n_classes-1)
    all_possible_labels = np.arange(len(tissue_encoder.classes_))
    class_names = [str(name) for name in tissue_encoder.classes_]
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f"\n{model_name} - Classification Report:")
    # Specify labels parameter to ensure all classes are included even if not present in test set
    print(classification_report(all_labels, all_preds, 
                              labels=all_possible_labels,
                              target_names=class_names,
                              zero_division=0))
    print(f"{model_name} - Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    return all_preds, all_labels, accuracy


## Detailed Evaluation


In [ ]:
# Load best model
checkpoint = torch.load('best_vae_transformer_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
condition_encoder.load_state_dict(checkpoint['condition_encoder_state_dict'])

# Evaluate model
transformer_preds, transformer_labels, transformer_acc = evaluate_model(
    model, condition_encoder, test_loader, "Transformer Multi-modal"
)


In [ ]:
# Confusion matrix
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
class_names = [str(name) for name in tissue_encoder.classes_]

cm = confusion_matrix(transformer_labels, transformer_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Transformer Multi-modal (GEX+TCR)\nAccuracy: {transformer_acc*100:.2f}%')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss comparison
axes[0].plot(combined_model_results['train_losses'], label='Train', alpha=0.7, linestyle='-')
axes[0].plot(combined_model_results['test_losses'], label='Test', alpha=0.7, linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(combined_model_results['train_accs'], label='Train', alpha=0.7, linestyle='-')
axes[1].plot(combined_model_results['test_accs'], label='Test', alpha=0.7, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
## Save Predictions and Model
# Create predictions dataframe and save
predictions_df = pd.DataFrame({
    'transformer_pred': transformer_preds,
    'true_label': transformer_labels
}, index=labels_filtered.index[test_idx])

predictions_df.to_csv('tissue_predictions_vae_transformer.csv', index=True)
print("Predictions saved to 'tissue_predictions_vae_transformer.csv'")

# Save final test accuracy
final_test_acc_df = pd.DataFrame({
    'Model': ['Transformer Multi-modal (GEX+TCR)'],
    'Best_Test_Acc': [combined_model_results['best_test_acc']],
    'Final_Test_Acc': [transformer_acc * 100]
})

# Save dataframe to CSV
acc_filepath = os.path.join(data_folder, 'final_test_accuracies_transformer.csv')
final_test_acc_df.to_csv(acc_filepath, index=False)
print(f"\nFinal Test Accuracies saved to '{acc_filepath}'")
print("\nFinal Test Accuracies:")
print(final_test_acc_df.to_string(index=False))
